In [1]:
# --- TEPAV Analytics Setup ---
import os
import pandas as pd
from analytics_helper import AnalyticsHelper
from google.analytics.data_v1beta.types import DateRange

# Initialize API
print("Initializing Analytics Client...")
analytics = AnalyticsHelper()
print("Ready!")

Initializing Analytics Client...
Ready!


In [2]:
# --- 2025 CİHAZ KULLANIM ANALİZİ (MOBİL/TABLET/DESKTOP) ---
print("\nFetching 2025 Device Usage Data...")

try:
    # Fetch Data
    df_devices = analytics.run_report(
        dimensions=["deviceCategory"],
        metrics=["activeUsers", "screenPageViews"],
        date_range=DateRange(start_date="2025-01-01", end_date="2025-12-31")
    )
    
    # Display Table
    print("\n2025 Web Sitesi Cihaz Dağılımı:")
    print(df_devices)
    
    # Visualize
    import plotly.express as px
    if not df_devices.empty:
        # Convert metrics to numbers
        df_devices['activeUsers'] = df_devices['activeUsers'].astype(int)
        df_devices['screenPageViews'] = df_devices['screenPageViews'].astype(int)
        
        # Pie Chart
        fig = px.pie(df_devices, 
                     values='activeUsers', 
                     names='deviceCategory', 
                     title='2025 Ziyaretçi Cihaz Dağılımı (Kullanıcı Sayısı)',
                     color_discrete_sequence=px.colors.qualitative.Set3)
        fig.update_traces(textposition='inside', textinfo='percent+label')
        fig.show()
        
except Exception as e:
    print(f"An error occurred: {e}")


Fetching 2025 Device Usage Data...

2025 Web Sitesi Cihaz Dağılımı:
  deviceCategory  activeUsers  screenPageViews
0         mobile       142159           218864
1        desktop       119869           288123
2         tablet         1618             2984
3       smart tv            6               20


In [3]:
# --- SOSYAL MEDYA TRAFİK RAPORU (EKLENDİ) ---
print("Fetching Social Media Traffic Data for 2025...")
from IPython.display import display, HTML
from google.analytics.data_v1beta.types import FilterExpression, Filter

try:
    # 1. Fetch Social Traffic Metrics
    # We filter for 'Organic Social' or 'Referral' from known social sources if needed.
    # Using 'sessionDefaultChannelGroup' is standard.
    
    df_social = analytics.run_report(
        dimensions=['sessionDefaultChannelGroup'],
        metrics=['sessions', 'activeUsers', 'eventCount'],
        date_range=DateRange(start_date="2025-01-01", end_date="2025-12-31"),
        dimension_filter=FilterExpression(
            filter=Filter(
                field_name="sessionDefaultChannelGroup",
                string_filter=Filter.StringFilter(value="Organic Social")
            )
        )
    )
    
    # Extract values (sum if multiple rows, though usually one for this filter)
    if not df_social.empty:
        sessions = df_social['sessions'].astype(int).sum()
        users = df_social['activeUsers'].astype(int).sum()
        events = df_social['eventCount'].astype(int).sum()
    else:
        sessions = 0
        users = 0
        events = 0
        
    # 2. Render HTML Card (Mimicking the User's Image)
    # We substitute 'Followers' with 'Active Users' and 'Tweets' with 'Interactions' as GA4 doesn't track external social stats.
    
    html_card = f"""
    <div style="font-family: Arial, sans-serif; max-width: 600px; margin: 20px auto; text-align: center; border: 1px solid #e0e0e0; border-radius: 12px; padding: 30px; box-shadow: 0 4px 6px rgba(0,0,0,0.05); background-color: white;">
        <h2 style="margin: 0 0 30px 0; font-size: 18px; color: #222; font-weight: 800; letter-spacing: 0.5px;">
            SOSYAL MEDYADAN WEB SİTESİNE<br>YÖNLENDİRİLEN TRAFİK (2025)
        </h2>
        
        <div style="display: flex; justify-content: space-around; align-items: flex-start; flex-wrap: wrap; gap: 20px;">
            
            <!-- Sessions -->
            <div style="flex: 1; min-width: 150px;">
                <div style="font-size: 32px; margin-bottom: 10px;">🖥️</div>
                <div style="font-size: 24px; font-weight: bold; color: #333;">{sessions:,.0f}</div>
                <div style="font-size: 14px; color: #666;">Oturum Sayısı</div>
            </div>
            
            <!-- Users (Proxy for Followers) -->
            <div style="flex: 1; min-width: 150px;">
                <div style="font-size: 32px; margin-bottom: 10px;">👤</div>
                <div style="font-size: 24px; font-weight: bold; color: #333;">{users:,.0f}</div>
                <div style="font-size: 14px; color: #666;">Gelen Kullanıcı*</div>
            </div>
            
        </div>
        
        <div style="margin-top: 30px;">
             <!-- Events (Proxy for Tweets/Activity) -->
            <div style="display: inline-block;">
                <div style="font-size: 28px; margin-bottom: 5px;">⚡</div>
                <span style="font-size: 20px; font-weight: bold; color: #333;">{events:,.0f}</span>
                <span style="font-size: 14px; color: #666; margin-left: 5px;">Etkileşim Sayısı*</span>
            </div>
        </div>

        <p style="margin-top: 25px; font-size: 11px; color: #999; font-style: italic;">
            *Not: Google Analytics dış veri (Takipçi/Tweet sayısı) sunmaz. <br>
            Burada 'Web Sitesine Gelen Tekil Kullanıcı' ve 'Sitedeki Toplam Etkileşim' gösterilmektedir.
        </p>
    </div>
    """
    
    display(HTML(html_card))
    
except Exception as e:
    print(f"Error fetching social data: {e}")

Fetching Social Media Traffic Data for 2025...
